In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # project root

In [2]:
import rasterio
from pathlib import Path
import numpy as np

from spm import SpatialPolygonMerger, ModelConfig, YOLOModel, visualize
from spm.utils.profiling import size_it, time_it
from spm.preprocessing.image_processing import binary_mask_to_contours

from sahi.postprocess.combine import GreedyNMMPostprocess, NMMPostprocess
from sahi.postprocess.backends import set_postprocess_backend

from spatial_mask_merging.smm.predictions import SMMPrediction
from spatial_mask_merging.smm.smm import SpatialMaskMerger

from utils.adapters import PredictionAdapter, SAHIPrediction

## Run Full Inference Pipeline

In [3]:
model_path = "../runs/segment/yolo-seg-whu/weights/best.pt"
tiff_path = Path("../test_images/cropped_tif.tif")

In [4]:
# Inference configuration
tile_size = 1500
batch_size = 4
overlap = 0.2
device = "cuda"  # or "cpu"

In [5]:
config = ModelConfig(
    model_path=model_path,
    tile_size=tile_size,
    batch_size=batch_size,
    overlap=overlap,
    device=device,
    )

In [6]:
model = YOLOModel(config)

In [ ]:
prediction = model(
                    tiff_path,
                    merge=False,
                    get_seg_from_binary_mask=True
                    )

[2026-06-30 02:05:18.379249] INFO - Performing prediction on image: ../test_images/cropped_tif.tif (width: 3822, height: 3658) (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/models/yolo.py:85:predict())


[2026-06-30 02:05:19.751178] INFO - Total tiles processed: 9 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/models/yolo.py:180:predict())


## Merge Predictions with SPM

In [22]:
smp = SpatialPolygonMerger()
smp.index(prediction)
spm_merged_prediction = smp.merge()
spm_merged_prediction.save_to_file(format="gpkg")

[2026-06-30 02:09:35.434470] INFO - Indexing polygons for merging (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/merging/spm.py:22:index())
[2026-06-30 02:09:35.435193] INFO - Execution time for index(): 00:00:00.000712 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/utils/profiling.py:18:wrapper())
[2026-06-30 02:09:35.577237] INFO - Starting merge process with 271 polygons. (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/merging/spm.py:126:merge())
[2026-06-30 02:09:35.629815] INFO - Total merged polygons: 74 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/merging/spm.py:194:merge())
[2026-06-30 02:09:35.630393] INFO - Total unmerged polygons: 86 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/merging/spm.py:196:merge())
[2026-06-30 02:09:35.630760] INFO - Total polygons after merging: 160 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/merging/spm.py:198:merge())
[2026-06-30 02:09:35.631118] INFO - 

In [9]:
# Visualize the SPM merged predictions on the original TIFF image
viz_dir = Path("predictions/viz")
viz_dir.mkdir(parents=True, exist_ok=True)
output_path = viz_dir / f"{tiff_path.stem}_spm_prediction.png"
with rasterio.open(tiff_path) as src:
    visualize(src, spm_merged_prediction, output_path)

[2026-06-30 02:06:27.772404] INFO - Visualizing predictions and saving to predictions/viz/cropped_tif_spm_prediction.png (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/visualization/overlays.py:19:visualize())


In [10]:
# Adapt the unmerged prediction to support both SAHI and SMM formats
prediction_adapter = PredictionAdapter(prediction)
unmerged_sahi_predictions = prediction_adapter.sahi
unmerged_smm_predictions = prediction_adapter.smm

## Merge Predictions with SAHI(GreedyNMM)

In [11]:
postprocess = GreedyNMMPostprocess(
        match_threshold=0.1,
        match_metric="IOS",
        class_agnostic=False,
    )

In [12]:
# Set SAHI backend to numpy to process predictions on CPU
set_postprocess_backend("numpy")

In [13]:
@size_it
@time_it
def _postprocess(sahi_predictions):
    merged_sahi_predictions = postprocess(sahi_predictions.predictions)
    return merged_sahi_predictions

In [14]:
# Postprocess the unmerged SAHI predictions to merge overlapping polygons
merged_sahi_predictions = _postprocess(unmerged_sahi_predictions)

[2026-06-30 02:06:46.782841] INFO - Execution time for _postprocess(): 00:00:00.146239 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/utils/profiling.py:18:wrapper())
[2026-06-30 02:06:47.124950] INFO - Execution time for _postprocess(): 00:00:00.286880 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/utils/profiling.py:18:wrapper())
[2026-06-30 02:06:47.156536] INFO - Peak memory usage for _postprocess(): 2.01 MB (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/utils/profiling.py:33:wrapper())


In [ ]:
# Convert merged SAHI predictions back to SPM format
sahi_prediction = SAHIPrediction()
sahi_prediction.predictions = merged_sahi_predictions
sahi_merged_predictions = PredictionAdapter(sahi_prediction).spm

In [ ]:
# Visualize the GNMM merged predictions on the original TIFF image
output_path = viz_dir / f"{tiff_path.stem}_sahi_gnmm_prediction.png"
with rasterio.open(tiff_path) as src:
    visualize(src, sahi_merged_predictions, output_path)

[2026-06-30 02:06:48.626163] INFO - Visualizing predictions and saving to predictions/viz/cropped_tif_sahi_gnmm_prediction.png (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/visualization/overlays.py:19:visualize())


## Merege Predictions with SMM

In [17]:
@size_it
@time_it
def smm_merger(prediction: SMMPrediction, image_size_hw: tuple[int, int]):
    # Initialize SMM with paper parameters
    merger = SpatialMaskMerger(
        tau_d=5.0,      # Distance threshold (pixels)
        tau_i=0.5,       # IoU threshold
        rho=10.0,        # R-tree search radius (pixels)
        beta1=0.3,       # Distance weight
        beta2=0.5,       # IoU weight
        beta3=0.2,       # Confidence weight
        gamma=0.5,       # Anti-chaining threshold
        lambda_=1.0      # Clustering penalty
    )

    return merger.merge(prediction, image_size_hw=image_size_hw)

In [18]:
smm_merged_obj = smm_merger(unmerged_smm_predictions, image_size_hw=prediction.image_shape)

[2026-06-30 02:07:54.510402] INFO - Execution time for smm_merger(): 00:00:41.878195 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/utils/profiling.py:18:wrapper())
[2026-06-30 02:07:54.539419] INFO - Peak memory usage for smm_merger(): 5923.47 MB (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/utils/profiling.py:33:wrapper())


In [19]:
# Convert SMM obj to SMMPrediction
smm_merged_predictions = SMMPrediction(image_name=tiff_path)

for pred in smm_merged_obj:

    contour = binary_mask_to_contours(pred["mask"].astype(np.uint8), normalize=False)
    segmentation = np.array(contour).reshape(1, -1, 2).tolist()

    smm_merged_predictions.add_annotation(
        type=pred["label"],
        class_id=pred["label"],
        confidence=pred["score"],
        segmentation=segmentation,
        bbox=pred["bbox"]
    )

In [20]:
# Convert merged SMM predictions back to SPM format
smm_merged_predictions_to_spm = PredictionAdapter(smm_merged_predictions).spm

In [21]:
# Visualize the SMM merged predictions on the original TIFF image
output_path = viz_dir / f"{tiff_path.stem}_smm_prediction.png"
with rasterio.open(tiff_path) as src:
    visualize(src, smm_merged_predictions_to_spm, output_path)

[2026-06-30 02:07:55.416995] INFO - Visualizing predictions and saving to predictions/viz/cropped_tif_smm_prediction.png (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/visualization/overlays.py:19:visualize())
